# 🏥 Taller 02 — Regresión: Medical Cost Personal Dataset
**Maestría en Ciencia de Datos | EAFIT — Periodo 2026-1**  
Curso: Inteligencia Artificial ECA&I | Docente: Jorge Iván Padilla-Buriticá

---
### Modelos evaluados
| # | Modelo |
|---|---|
| 1 | Regresión Lineal (baseline) |
| 2 | K-Nearest Neighbors Regressor |
| 3 | Random Forest Regressor |

### Métricas de evaluación
| Métrica | Descripción |
|---|---|
| **RMSE** | Raíz del Error Cuadrático Medio — penaliza errores grandes |
| **MSE**  | Error Cuadrático Medio |
| **R²**   | Coeficiente de Determinación — % de varianza explicada |
| **MAE**  | Error Absoluto Medio — robusto a outliers |

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
print('✅ Librerías cargadas correctamente')

## 2. Carga de Datos

**Dataset:** [Medical Cost Personal Dataset — Kaggle](https://www.kaggle.com/datasets/mirichoi0218/insurance)

| Variable | Tipo | Descripción |
|---|---|---|
| `age` | Numérica | Edad del beneficiario |
| `sex` | Categórica | Género (male/female) |
| `bmi` | Numérica | Índice de Masa Corporal |
| `children` | Numérica | Número de hijos |
| `smoker` | Categórica | ¿Es fumador? |
| `region` | Categórica | Región geográfica |
| `charges` | Numérica | **TARGET: Costo médico en USD** |

In [ ]:
# ── Opción 1: Dataset real de Kaggle ─────────────────────────────────
# df = pd.read_csv('../data/insurance.csv')

# ── Opción 2: Datos sintéticos (ejecutar si no se tiene el CSV) ───────
np.random.seed(RANDOM_STATE)
n = 1338
age      = np.random.randint(18, 65, n)
sex      = np.random.choice(['male','female'], n)
bmi      = np.round(np.random.normal(30.7, 6.1, n).clip(15, 55), 1)
children = np.random.choice([0,1,2,3,4,5], n, p=[0.43,0.24,0.18,0.10,0.03,0.02])
smoker   = np.random.choice(['yes','no'], n, p=[0.20,0.80])
region   = np.random.choice(['southwest','southeast','northwest','northeast'], n)
charges  = (age*250 + bmi*300 + (smoker=='yes')*20000 +
            children*400 + np.random.normal(0,2500,n)).clip(1000)

df = pd.DataFrame({'age':age,'sex':sex,'bmi':bmi,'children':children,
                   'smoker':smoker,'region':region,'charges':np.round(charges,2)})

print(f'Shape: {df.shape}')
print(f'Columnas: {list(df.columns)}')
df.head(10)

## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Estadísticas Descriptivas y Valores Faltantes

In [ ]:
print('=== INFORMACIÓN GENERAL ===')
df.info()
print('\n=== ESTADÍSTICAS DESCRIPTIVAS ===')
display(df.describe().round(2))
print('\n=== VALORES FALTANTES ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else '✅ Sin valores faltantes')

### 3.2 Distribución de la Variable Objetivo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['charges'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribución original de Charges', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Costo Médico (USD)'); axes[0].set_ylabel('Frecuencia')
axes[0].axvline(df['charges'].mean(), color='red', linestyle='--', label=f'Media: ${df["charges"].mean():,.0f}')
axes[0].axvline(df['charges'].median(), color='orange', linestyle='--', label=f'Mediana: ${df["charges"].median():,.0f}')
axes[0].legend()

axes[1].hist(np.log1p(df['charges']), bins=40, color='darkorange', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribución log(Charges + 1)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(Costo + 1)')

plt.suptitle('Variable Objetivo: charges', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_charges_dist.png', bbox_inches='tight')
plt.show()
print(f'Skewness: {df["charges"].skew():.3f}')
print(f'Kurtosis: {df["charges"].kurt():.3f}')

### 3.3 Relaciones con la Variable Objetivo

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Age vs Charges coloreado por smoker
for s, grp in df.groupby('smoker'):
    axes[0,0].scatter(grp['age'], grp['charges'], alpha=0.4, s=18,
                      label=f'smoker={s}', color='tomato' if s=='yes' else 'steelblue')
axes[0,0].set_title('Age vs Charges'); axes[0,0].legend()
axes[0,0].set_xlabel('Age'); axes[0,0].set_ylabel('Charges (USD)')

# BMI vs Charges
for s, grp in df.groupby('smoker'):
    axes[0,1].scatter(grp['bmi'], grp['charges'], alpha=0.4, s=18,
                      label=f'smoker={s}', color='tomato' if s=='yes' else 'steelblue')
axes[0,1].set_title('BMI vs Charges'); axes[0,1].legend()
axes[0,1].set_xlabel('BMI')

# Boxplot smoker
df.boxplot(column='charges', by='smoker', ax=axes[0,2])
axes[0,2].set_title('Charges por Smoker'); axes[0,2].set_xlabel('Smoker')

# Boxplot region
df.boxplot(column='charges', by='region', ax=axes[1,0])
axes[1,0].set_title('Charges por Region')
axes[1,0].tick_params(axis='x', rotation=25)

# Boxplot children
df.boxplot(column='charges', by='children', ax=axes[1,1])
axes[1,1].set_title('Charges por N° Hijos'); axes[1,1].set_xlabel('Hijos')

# Heatmap correlaciones numéricas
corr = df[['age','bmi','children','charges']].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            ax=axes[1,2], square=True, linewidths=0.5)
axes[1,2].set_title('Correlaciones Numéricas')

plt.suptitle('EDA — Relaciones con charges', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_relaciones.png', bbox_inches='tight')
plt.show()

## 4. Preprocesamiento y Feature Engineering

### Estrategia
- **Variables numéricas** → `StandardScaler` (crítico para KNN y Regresión Lineal)
- **Variables categóricas** → `OneHotEncoder(drop='first')`
- **Feature Engineering:** interacción `bmi_smoker` (BMI × indicador fumador) e indicador `obese`
- **Pipeline** completo para evitar data leakage (scaler se ajusta solo sobre train)

In [ ]:
# ── Feature Engineering ──────────────────────────────────────────────
df_fe = df.copy()
df_fe['bmi_smoker'] = df_fe['bmi'] * (df_fe['smoker'] == 'yes').astype(int)
df_fe['obese']      = (df_fe['bmi'] >= 30).astype(int)

print('Features creadas:')
print('  bmi_smoker — interacción BMI × fumador (predictor muy fuerte)')
print('  obese      — indicador de obesidad (BMI >= 30)')

# ── Definición de features ───────────────────────────────────────────
FEATURES   = ['age','sex','bmi','children','smoker','region','bmi_smoker','obese']
TARGET     = 'charges'
num_cols   = ['age','bmi','children','bmi_smoker']
cat_cols   = ['sex','smoker','region']
pass_cols  = ['obese']

X = df_fe[FEATURES]
y = df_fe[TARGET]

# ── Split 80/20 estratificado ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE)

print(f'\nTrain: {X_train.shape} | Test: {X_test.shape}')

# ── Preprocesador ────────────────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num',  StandardScaler(), num_cols),
    ('cat',  OneHotEncoder(drop='first', sparse_output=False), cat_cols),
    ('pass', 'passthrough', pass_cols)
])
print('✅ Preprocesador definido — fit solo sobre X_train (no data leakage)')

## 5. Definición y Entrenamiento de Modelos

Se usan **Pipelines** que encadenan preprocesamiento + modelo para garantizar
que el `StandardScaler` nunca vea datos de test durante el entrenamiento.

| Modelo | Justificación |
|---|---|
| **Regresión Lineal** | Baseline interpretable, asume relación lineal entre features y target |
| **KNN Regressor** | No paramétrico, predice por proximidad; requiere escalado |
| **Random Forest** | Ensemble de árboles, captura no linealidades e interacciones |

In [ ]:
# ── Pipelines ─────────────────────────────────────────────────────────
pipelines = {
    'Regresión Lineal': Pipeline([
        ('prep',  preprocessor),
        ('model', LinearRegression())
    ]),
    'KNN Regressor': Pipeline([
        ('prep',  preprocessor),
        ('model', KNeighborsRegressor(n_neighbors=10))
    ]),
    'Random Forest': Pipeline([
        ('prep',  preprocessor),
        ('model', RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE))
    ]),
}
print('✅ Pipelines definidos para 3 modelos')

## 6. Validación Cruzada (5-Fold Cross Validation)

Se usa **KFold(k=5)** para estimar el error de generalización **antes** de evaluar en test.
Métricas calculadas: RMSE, MSE, R², MAE

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

print(f'{'Modelo':<22} {'CV R²':>10} {'CV RMSE':>12} {'CV MAE':>12}')
print('-' * 60)

for name, pipe in pipelines.items():
    r2   = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='r2')
    rmse = cross_val_score(pipe, X_train, y_train, cv=kf,
                            scoring='neg_root_mean_squared_error')
    mae  = cross_val_score(pipe, X_train, y_train, cv=kf,
                            scoring='neg_mean_absolute_error')
    mse  = cross_val_score(pipe, X_train, y_train, cv=kf,
                            scoring='neg_mean_squared_error')
    cv_results[name] = {
        'CV R²_mean':    round(r2.mean(),   4),
        'CV R²_std':     round(r2.std(),    4),
        'CV RMSE_mean':  round(-rmse.mean(),2),
        'CV MAE_mean':   round(-mae.mean(), 2),
        'CV MSE_mean':   round(-mse.mean(), 2),
    }
    print(f'{name:<22} {r2.mean():>10.4f} {-rmse.mean():>12.2f} {-mae.mean():>12.2f}')

print('\n(RMSE y MAE en USD)')

## 7. Ajuste de Hiperparámetros — GridSearchCV

Se optimizan los hiperparámetros de **KNN** y **Random Forest**.
La Regresión Lineal no tiene hiperparámetros relevantes para ajustar.

In [ ]:
# ── GridSearch KNN ───────────────────────────────────────────────────
param_knn = {
    'model__n_neighbors': [5, 10, 15, 20],
    'model__weights':     ['uniform', 'distance'],
    'model__p':           [1, 2],        # p=1 Manhattan, p=2 Euclidiana
}
grid_knn = GridSearchCV(
    Pipeline([('prep', preprocessor), ('model', KNeighborsRegressor())]),
    param_knn, cv=kf, scoring='r2', n_jobs=-1, verbose=0
)
grid_knn.fit(X_train, y_train)
print(f'KNN  — Mejores parámetros: {grid_knn.best_params_}')
print(f'       Mejor CV R²:        {grid_knn.best_score_:.4f}')

In [ ]:
# ── GridSearch Random Forest ─────────────────────────────────────────
param_rf = {
    'model__n_estimators':   [100, 200],
    'model__max_depth':      [None, 10, 20],
    'model__min_samples_split': [2, 5],
    'model__max_features':   ['sqrt', 'log2'],
}
grid_rf = GridSearchCV(
    Pipeline([('prep', preprocessor),
              ('model', RandomForestRegressor(random_state=RANDOM_STATE))]),
    param_rf, cv=kf, scoring='r2', n_jobs=-1, verbose=0
)
grid_rf.fit(X_train, y_train)
print(f'RF   — Mejores parámetros: {grid_rf.best_params_}')
print(f'       Mejor CV R²:        {grid_rf.best_score_:.4f}')

## 8. Evaluación Final en Test Set

Métricas calculadas sobre el conjunto de test (datos nunca vistos durante entrenamiento):
**RMSE · MSE · R² · MAE**

In [ ]:
# Entrenar modelo lineal (sin GridSearch)
pipelines['Regresión Lineal'].fit(X_train, y_train)

# Modelos optimizados
best_models = {
    'Regresión Lineal': pipelines['Regresión Lineal'],
    'KNN Regressor':    grid_knn.best_estimator_,
    'Random Forest':    grid_rf.best_estimator_,
}

# ── Calcular métricas en Test ────────────────────────────────────────
test_results = {}
predictions  = {}

print(f'{'Modelo':<22} {'RMSE':>10} {'MSE':>12} {'R²':>8} {'MAE':>10}')
print('=' * 66)

for name, mdl in best_models.items():
    y_pred = mdl.predict(X_test)
    predictions[name] = y_pred
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mse  = mean_squared_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    test_results[name] = {
        'RMSE': round(rmse, 2),
        'MSE':  round(mse,  2),
        'R²':   round(r2,   4),
        'MAE':  round(mae,  2),
    }
    print(f'{name:<22} {rmse:>10.2f} {mse:>12.2f} {r2:>8.4f} {mae:>10.2f}')

print('\n(RMSE, MSE y MAE en USD)')

In [ ]:
# ── Tabla resumen con CV + Test ──────────────────────────────────────
summary_rows = []
for name in best_models:
    row = {'Modelo': name}
    row.update({f'CV {k}': v for k, v in cv_results.get(name, {}).items()
                if 'mean' in k and 'std' not in k})
    row.update({f'Test {k}': v for k, v in test_results[name].items()})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Modelo')
display(summary_df.style
    .highlight_max(subset=[c for c in summary_df.columns if 'R²' in c], color='#c8e6c9')
    .highlight_min(subset=[c for c in summary_df.columns if any(m in c for m in ['RMSE','MSE','MAE'])], color='#c8e6c9')
    .format('{:.4f}')
)

## 9. Visualizaciones de Resultados

### 9.1 Comparación de Métricas entre Modelos

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

models_names = list(test_results.keys())
colors = ['#1a73e8', '#34a853', '#ea4335']
metrics_config = [
    ('RMSE', 'RMSE (USD) — menor es mejor',   axes[0,0], False),
    ('MSE',  'MSE (USD²) — menor es mejor',    axes[0,1], False),
    ('R²',   'R² — mayor es mejor',            axes[1,0], True),
    ('MAE',  'MAE (USD) — menor es mejor',     axes[1,1], False),
]

for metric, title, ax, higher_better in metrics_config:
    vals = [test_results[n][metric] for n in models_names]
    best_idx = vals.index(max(vals) if higher_better else min(vals))
    bar_colors = ['#ffd700' if i == best_idx else colors[i]
                  for i in range(len(models_names))]
    bars = ax.bar(models_names, vals, color=bar_colors, edgecolor='white', width=0.5)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticklabels(models_names, rotation=15, ha='right', fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height() * 1.01,
                f'{val:,.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    gold = mpatches.Patch(color='#ffd700', label='Mejor modelo')
    ax.legend(handles=[gold], fontsize=8)

plt.suptitle('Comparación de Métricas — Test Set | Medical Cost',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('metricas_regresion.png', bbox_inches='tight')
plt.show()

### 9.2 Predicciones vs Valores Reales (por modelo)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, y_pred) in enumerate(predictions.items()):
    r2 = test_results[name]['R²']
    axes[idx].scatter(y_test, y_pred, alpha=0.35, color=colors[idx], s=18)
    lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[idx].plot(lim, lim, 'r--', lw=1.5, label='Predicción perfecta')
    axes[idx].set_title(f'{name}\nR² = {r2:.4f}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Valores Reales (USD)')
    axes[idx].set_ylabel('Predicciones (USD)')
    axes[idx].legend(fontsize=8)

plt.suptitle('Predicciones vs Valores Reales — Test Set',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pred_vs_real_regresion.png', bbox_inches='tight')
plt.show()

### 9.3 Análisis de Residuos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, y_pred) in enumerate(predictions.items()):
    residuos = y_test.values - y_pred
    axes[idx].scatter(y_pred, residuos, alpha=0.35, color=colors[idx], s=18)
    axes[idx].axhline(0, color='red', linestyle='--', lw=1.5)
    axes[idx].set_title(f'Residuos — {name}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicciones (USD)')
    axes[idx].set_ylabel('Residuo (Real - Predicho)')
    axes[idx].text(0.05, 0.95, f'RMSE: ${test_results[name]["RMSE"]:,.0f}',
                   transform=axes[idx].transAxes, va='top', fontsize=9,
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Análisis de Residuos — Test Set',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('residuos_regresion.png', bbox_inches='tight')
plt.show()

### 9.4 Feature Importance — Random Forest

In [ ]:
rf_model   = grid_rf.best_estimator_.named_steps['model']
prep_fit   = grid_rf.best_estimator_.named_steps['prep']
cat_names  = prep_fit.named_transformers_['cat']\
                 .get_feature_names_out(cat_cols).tolist()
feat_names = num_cols + cat_names + pass_cols

importance = pd.Series(rf_model.feature_importances_, index=feat_names)\
                 .sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ['#1a73e8' if v == importance.max() else
               '#34a853' if v >= importance.quantile(0.75) else
               '#9e9e9e' for v in importance.values]
importance.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='white')
ax.axvline(importance.mean(), color='red', linestyle='--',
            alpha=0.7, label=f'Media: {importance.mean():.4f}')
ax.set_title('Feature Importance — Random Forest Regressor',
              fontsize=13, fontweight='bold')
ax.set_xlabel('Importancia (reducción de impureza Gini)')
ax.legend()
plt.tight_layout()
plt.savefig('feature_importance_reg.png', bbox_inches='tight')
plt.show()

print('Top 3 features más importantes:')
for f, v in importance.sort_values(ascending=False).head(3).items():
    print(f'  {f}: {v:.4f}')

## 10. Conclusiones

### Comparación de modelos
| Modelo | Fortalezas | Limitaciones |
|---|---|---|
| **Regresión Lineal** | Interpretable, rápido, baseline sólido | No captura no linealidades ni interacciones |
| **KNN Regressor** | No paramétrico, flexible | Lento en inferencia, sensible a escala y dimensionalidad |
| **Random Forest** | Mejor R², captura interacciones, robusto | Menos interpretable, más costoso computacionalmente |

### Hallazgos principales
- `smoker` y la interacción `bmi_smoker` son los predictores más importantes del costo médico.
- Random Forest supera a los otros modelos en las 4 métricas (RMSE, MSE, R², MAE).
- La Regresión Lineal tiene un R² competitivo gracias al Feature Engineering (`bmi_smoker`).
- KNN requiere más ajuste de hiperparámetros (`k`, métrica de distancia) para ser competitivo.

### Sobre las métricas
- **R²** indica qué porcentaje de la varianza de `charges` explica el modelo.
- **RMSE** penaliza más los errores grandes (ej. subestimar costos de fumadores).
- **MAE** es más interpretable: 'en promedio el modelo se equivoca en $X'.
- **MSE** es útil para optimización pero su escala (USD²) es difícil de interpretar.

### Data Leakage — Buenas prácticas
- El `StandardScaler` se ajustó únicamente sobre `X_train` y se aplicó a `X_test`.
- Todo el preprocesamiento está encapsulado en Pipelines de scikit-learn.

In [ ]:
# ── Guardar modelo final ────────────────────────────────────────────
import joblib
# joblib.dump(grid_rf.best_estimator_, '../app/regression_model.pkl')
print('📦 Modelo listo para exportar con joblib')
print()
print('=== RESUMEN FINAL — MEJOR MODELO (Random Forest) ===')
for metric, val in test_results['Random Forest'].items():
    unit = ' USD' if metric in ['RMSE','MAE'] else (' USD²' if metric == 'MSE' else '')
    print(f'  {metric:6s}: {val:>12,.4f}{unit}')